In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 09 · Inferencia — del modelo registrado a la app

Se exporta el `champion` a `models/model.joblib` (la app lo carga del repo, **sin** consultar Databricks) y se genera
`models/input_schema.json`: columnas, rangos posibles, categorías conocidas, umbral y textos de la app.

In [2]:
import json
import joblib
os.environ.setdefault("MLFLOW_ALLOW_PICKLE_DESERIALIZATION", "true")
from proyecto.config import REPORTS_DIR, PLAUSIBLE_RANGE, TARGET
from proyecto.models.train import split_xy
from proyecto.inference.schema import build_schema, save_schema
from proyecto.inference.predict import InputValidationError, predict_with_scores, load_model

registro = json.loads((REPORTS_DIR / "08_registro.json").read_text(encoding="utf-8"))
resultados = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
train = joblib.load(ROOT / "data" / "processed" / "train.joblib")
cols = registro["columnas_entrada"]
try:
    import mlflow
    from proyecto.tracking.mlflow_utils import configure_mlflow
    configure_mlflow()
    modelo = mlflow.sklearn.load_model(f"models:/{registro['modelo_registrado']}@champion")
    fuente = f"registro: {registro['modelo_registrado']} v{registro['version']} @champion"
except Exception as error:
    modelo = joblib.load(ROOT / "models" / "final_pipeline.joblib")
    fuente = f"copia local de la etapa 7 (registro no disponible: {str(error)[:80]})"
from proyecto.models.portable import make_portable
# XGBoost se guarda como JSON (portable entre Linux y Windows) y sin compresión; predicciones idénticas
joblib.dump(make_portable(modelo), ROOT / "models" / "model.joblib", compress=0)
print("exportado models/model.joblib desde", fuente)

exportado models/model.joblib desde registro: libranzaradar_mora30 v1 @champion


In [3]:
NUMERICAS = [c for c in cols if pd.api.types.is_numeric_dtype(train[c])]
CATEGORICAS = [c for c in cols if c not in NUMERICAS]
ETIQUETAS = {
    "term_months": "Plazo (meses)", "loan_amount": "Monto del crédito ($)", "monthly_installment": "Cuota mensual ($)",
    "interest_rate_ea": "Tasa efectiva anual (%)", "due_day": "Día de pago de la cuota", "has_guarantee": "¿Tiene garantía? (1 = sí)",
    "guarantee_value": "Valor de la garantía ($)", "has_cosigner": "¿Tiene codeudor? (1 = sí)", "product": "Producto",
    "business_type": "Tipo de negocio", "economic_destination": "Destino económico", "city": "Ciudad del cliente",
}
umbral = resultados["decisiones"]["umbral"]
tipico = train[cols].iloc[0].to_dict()
ejemplos = {
    "Crédito típico (36 meses)": {**tipico, "term_months": 36, "loan_amount": 9_000_000, "monthly_installment": 330_000},
    "Plazo corto (6 meses)": {**tipico, "term_months": 6, "loan_amount": 3_000_000, "monthly_installment": 540_000},
}
schema = build_schema(train, NUMERICAS, CATEGORICAS, ["al día", "mora > 30 días"],
                      valid_range={c: PLAUSIBLE_RANGE[c] for c in NUMERICAS if c in PLAUSIBLE_RANGE}, labels=ETIQUETAS)
schema.update({
    "threshold": float(umbral), "base_rate": float(train[TARGET].mean()), "review_margin": 0.005,
    "mode": {c: str(train[c].mode()[0]) for c in CATEGORICAS},
    "examples": {k: {c: (v.item() if hasattr(v, "item") else v) for c, v in e.items()} for k, e in ejemplos.items()},
    "app_title": "LibranzaRadar · Priorización de seguimiento de créditos",
    "model_version": f"{registro['modelo_registrado']} v{registro['version']} (@champion)",
    "test_metrics": {"recall": resultados["prueba"]["recall de mora (métrica principal)"]["modelo"],
                     "precision": resultados["prueba"]["precisión (exigida ≥ 20 %)"]["modelo"],
                     "auc": resultados["prueba"]["AUC"]["modelo"], "flag_rate": resultados["porcentaje_marcado"]["modelo"]},
})
save_schema(schema, ROOT / "models" / "input_schema.json")
print("esquema:", len(NUMERICAS), "numéricas,", len(CATEGORICAS), "categóricas · umbral", round(umbral, 4))

esquema: 8 numéricas, 4 categóricas · umbral 0.301


## Probar la predicción: normal, incompleta e imposible

In [4]:
normal = pd.DataFrame([schema["examples"]["Crédito típico (36 meses)"]])[cols]
print(predict_with_scores(normal, modelo, schema).round(3).T)
incompleta = normal.copy(); incompleta["interest_rate_ea"] = np.nan; incompleta["city"] = np.nan
print("incompleta ->", predict_with_scores(incompleta, modelo, schema)["decision"].iloc[0])
imposible = normal.copy(); imposible["term_months"] = -5
try:
    predict_with_scores(imposible, modelo, schema)
except InputValidationError as error:
    print("imposible -> rechazada:", error)

                           0
decision              al día
proba_al día           0.843
proba_mora > 30 días   0.157
most_likely           al día
needs_review           False
relative_risk          0.994
incompleta -> al día
imposible -> rechazada: term_months: debe estar entre 1 y 360


## El archivo exportado es el modelo que se evaluó

In [5]:
test = joblib.load(ROOT / "data" / "processed" / "test.joblib")
exportado = load_model(ROOT / "models" / "model.joblib")
p = exportado.predict_proba(test[cols])[:, 1]
recall = (p[test[TARGET] == 1] >= umbral).mean()
esperado = resultados["prueba"]["recall de mora (métrica principal)"]["modelo"][0]
print(f"recall con el exportado: {recall:.6f} · en la etapa 7: {esperado:.6f}")
assert abs(recall - esperado) < 1e-9, "el modelo exportado no es el que se evaluó"

recall con el exportado: 0.000000 · en la etapa 7: 0.000000


In [6]:
ficha = (ROOT / "models" / "ficha_modelo.md").read_text(encoding="utf-8")
print("ficha_modelo.md:", "completa" if "_______________" not in ficha else f"{ficha.count('_______________')} campos en blanco")

ficha_modelo.md: completa
